In [1]:
import yaml

from _configs.country_config import *
from _configs.files_config import *
from _configs.run_config import *

from _utils.utils import *

In [2]:
output_path = Path(validation_path) / "output"

input_data = yaml.safe_load((Path(VALIDATION_RUN_INPUTS_DIR)/"input_validation.yml").read_text(encoding="utf-8"))

population_scale_from_input_yml = input_data["population_demographic"]["artificial_rescaling_of_population_size"]
starting_date = input_data["simulation_timeframe"]["starting_date"]
ending_date = input_data["simulation_timeframe"]["ending_date"]

info(f"Checking outputs for validation run: {validation_path}\n")
print(f"Population scale from input yml: {population_scale_from_input_yml}")
print(f"Starting date: {starting_date}")
print(f"Ending date: {ending_date}")

→ Checking outputs for validation run: validation_runs/validation_3_0.25_population_scale_one_pattern_20_replicates

Population scale from input yml: 0.25
Starting date: 2011/1/1
Ending date: 2024/1/1


In [3]:
output_1_db = output_path / "validation_monthly_data_3.db"
print(f"Reading monthly data from: {output_1_db}")
# date = get_table_with_columns(output_1_db, ["*"], "monthly_data")
date = get_table(output_1_db, "monthly_data")
# Load yaml input file
date['date'] = pd.to_datetime(starting_date) + pd.to_timedelta(date['days_elapsed'], unit='D')
date = date.rename(columns={"id": "monthly_data_id"})
date = date.sort_values("date")
date_dict = date.set_index("monthly_data_id")["date"].to_dict()

Reading monthly data from: validation_runs/validation_3_0.25_population_scale_one_pattern_20_replicates/output/validation_monthly_data_3.db


In [4]:
# Check the range of the date data from the db
print(f"Date range from db: {min(date_dict.values())} to {max(date_dict.values())}")

print(f"Date range from input yml: {starting_date} to {ending_date}")


db_min = min(date_dict.values())
db_max = max(date_dict.values())
expected_start = pd.to_datetime(starting_date)
expected_end = pd.to_datetime(ending_date)

mismatches = []
if db_min != expected_start:
    mismatches.append(
        f"STARTING DATE mismatch: db min = {db_min}, yml starting_date = {expected_start}"
    )
if db_max != expected_end:
    mismatches.append(
        f"ENDING DATE mismatch: db max = {db_max}, yml ending_date = {expected_end}"
    )

if mismatches:
    error(mismatches)
    raise ValueError(
        "Date range from db does not match date range from input yml:\n"
        + "\n".join(mismatches)
    )
else:
    ok("Date range from db matches date range from input yml.")

Date range from db: 2011-01-01 00:00:00 to 2016-05-01 00:00:00
Date range from input yml: 2011/1/1 to 2024/1/1
✗ ['ENDING DATE mismatch: db max = 2016-05-01 00:00:00, yml ending_date = 2024-01-01 00:00:00']


ValueError: Date range from db does not match date range from input yml:
ENDING DATE mismatch: db max = 2016-05-01 00:00:00, yml ending_date = 2024-01-01 00:00:00

In [5]:
log_path = Path(validation_path) / "log"

for log_file in log_path.glob("*.log"):
    with open(log_file, "r") as f:
        log_content = f.read()

        # print the last line of the log file
        last_line = log_content.strip().split("\n")[-1]
        print(f"Last line of log file {log_file.name}: {last_line}")
    

Last line of log file validation_rep_15.log: [2026-08-27 14:24:14] [info] Model time 1980, schedule recurrence event at time 1987, clinical end event at time 1989
Last line of log file validation_rep_6.log: [2026-08-27 14:24:25] [info] Model time 1962, schedule recurrence event at time 1969, clinical end event at time 1973
Last line of log file validation_rep_19.log: [2026-08-27 14:25:08] [info] Model time 2000, schedule recurrence event at time 2007, clinical end event at time 2008
Last line of log file validation_rep_16.log: [2026-08-27 14:24:57] [info] Day: 2070
Last line of log file validation_rep_18.log: [2026-08-27 14:24:34] [info] Model time 1962, schedule recurrence event at time 1969, clinical end event at time 1970
Last line of log file validation_rep_3.log: [2026-08-27 14:24:24] [info] Model time 1965, schedule recurrence event at time 1972, clinical end event at time 1973
Last line of log file validation_rep_13.log: [2026-08-27 14:24:45] [info] Model time 1991, schedule rec